# Лабораторная работа №12. DNS: имя становится адресом

Локальный authoritative BIND: A/AAAA/CNAME/NS, request/reply по UDP/53 и тот же DNS message через TCP.

## Схема

![DNS](assets/dns.svg)

`pc1` — client, `dns1` — authoritative server `lab.example`. На lab-link server имеет `10.12.0.53/24`.

## 1. Подготовим client

На `pc1`:

```bash
ip addr flush dev eth1
ip addr add 10.12.0.10/24 dev eth1
ping -c 1 10.12.0.53
```

DNS debugging бессмысленен, если сначала не подтверждена обычная IP-связность.

### Q1. Что именно получает приложение от DNS: «соединение с сайтом» или данные, которые приложение потом использует для следующего сетевого шага?

`введите ваш ответ`

## 2. A, AAAA, CNAME и NS

На `pc1`:

```bash
dig @10.12.0.53 web.lab.example A
dig @10.12.0.53 web.lab.example AAAA
dig @10.12.0.53 alias.lab.example CNAME
dig @10.12.0.53 lab.example NS
```

Смотрите sections QUESTION/ANSWER/AUTHORITY, flags и TTL записи.

### Q2. Чем CNAME отличается от A/AAAA? Почему alias не обязан содержать IP-адрес прямо в своей записи?

`введите ваш ответ`

In [ ]:
%%capture_traffic pc1:eth1 --filter "udp.port == 53 or tcp.port == 53" --timeout 12 --packets 100 --save captures/dns.pcap
import time
print("На pc1 выполните dig @10.12.0.53 web.lab.example A")
time.sleep(10)

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
viewer = PcapViewer("captures/dns.pcap", "dns", limit=40)
frames = viewer.packets()
display(frames)
if not frames.empty:
    for n in frames["№"].head(4):
        print("\nframe", n)
        print("\n".join(viewer.packet(int(n), "protocols").splitlines()[:80]))

## 3. DNS обычно UDP, но не только UDP

Теперь принудительно используйте TCP:

```bash
dig +tcp @10.12.0.53 web.lab.example A
```

Повторите capture с тем же filter и найдите TCP handshake перед DNS request/response.

### Q3. Почему корректнее говорить «DNS обычно использует UDP/53», а не «DNS работает по UDP»?

`введите ваш ответ`

## 4. Что делает resolver

`dig` здесь обращается прямо к указанному server. Обычное приложение чаще вызывает системный resolver, который читает resolver configuration, использует cache и может обращаться к recursive resolver вместо authoritative server.

## CHECK

Checker с `pc1` выполнит реальные queries к `dns1` и проверит A/CNAME ответы.

## Итог

DNS message — application protocol поверх transport. Он преобразует names в typed records; дальнейший TCP/UDP/HTTP шаг выполняется уже после получения ответа.